In [ ]:
# 03 — Análisis estadístico e inferencia

## Objetivo

En el análisis exploratorio se identificaron diferencias entre los clientes
clasificados como `BAD = 0` y `BAD = 1` en distintas variables financieras
y crediticias.

En este notebook se aplicarán técnicas de inferencia estadística para evaluar
si algunas de estas diferencias presentan evidencia estadística suficiente.

El análisis incluirá:

- Formulación de hipótesis nula y alternativa.
- Selección de pruebas estadísticas según las características de los datos.
- Interpretación de p-values.
- Análisis del tamaño del efecto.
- Estudio de asociaciones entre variables.

Se utilizará un nivel de significación de α = 0,05.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import stats

df = pd.read_csv(r"C:\Users\Sistemas\Desktop\BOOTCAMP DATA\Master Data Sciencie & AI\Proyecto banking\hmeq.csv")
df.shape


(5960, 13)

In [3]:
df.head()

,BAD,LOAN,MORTDUE,VALUE,REASON,JOB,YOJ,DEROG,DELINQ,CLAGE,NINQ,CLNO,DEBTINC
0,1,1100,25860.0,39025.0,HomeImp,Other,10.5,0.0,0.0,94.366667,1.0,9.0,NaN
1,1,1300,70053.0,68400.0,HomeImp,Other,7.0,0.0,2.0,121.833333,0.0,14.0,NaN
2,1,1500,13500.0,16700.0,HomeImp,Other,4.0,0.0,0.0,149.466667,1.0,10.0,NaN
3,1,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,1700,97800.0,112000.0,HomeImp,Office,3.0,0.0,0.0,93.333333,0.0,14.0,NaN


In [ ]:
## 3. Contraste de hipótesis: CLAGE según BAD

Durante el EDA se observaron valores inferiores de `CLAGE` entre los clientes
clasificados como `BAD = 1`.

Se plantea el siguiente contraste:

**H₀ (hipótesis nula):** no existen diferencias en la distribución de `CLAGE`
entre los grupos `BAD = 0` y `BAD = 1`.

**H₁ (hipótesis alternativa):** existen diferencias en la distribución de
`CLAGE` entre los grupos `BAD = 0` y `BAD = 1`.

Se utilizará un nivel de significación de α = 0,05.

In [4]:
clage_0 = df.loc[df["BAD"] == 0, "CLAGE"].dropna()
clage_1 = df.loc[df["BAD"] == 1, "CLAGE"].dropna()

print("BAD = 0:", len(clage_0))
print("BAD = 1:", len(clage_1))

BAD = 0: 4541
BAD = 1: 1111


In [5]:
print("CLAGE - BAD = 0")
print(clage_0.describe())

print("\nCLAGE - BAD = 1")
print(clage_1.describe())

CLAGE - BAD = 0
count    4541.000000
mean      187.002355
std        84.465217
min         0.486711
25%       120.219885
50%       180.415787
75%       240.157802
max       649.747104
Name: CLAGE, dtype: float64

CLAGE - BAD = 1
count    1111.000000
mean      150.190183
std        84.952286
min         0.000000
25%        96.033333
50%       132.866667
75%       193.283333
max      1168.233561
Name: CLAGE, dtype: float64


In [6]:
print("Asimetría BAD = 0:", stats.skew(clage_0))
print("Asimetría BAD = 1:", stats.skew(clage_1))

Asimetría BAD = 0: 0.9011613884349553
Asimetría BAD = 1: 3.4761980179761083


In [ ]:
### 3.1 Selección de la prueba estadística

Los grupos presentan tamaños de muestra elevados, pero la distribución de
`CLAGE` muestra asimetría, especialmente en `BAD = 1`, donde se obtiene
una asimetría de 3,48 y se observan valores extremos.

Por este motivo se utilizará la prueba no paramétrica de Mann–Whitney U
para evaluar si existen diferencias en la distribución de `CLAGE` entre
los dos grupos.

Hipótesis:

- H₀: las distribuciones de `CLAGE` no presentan diferencias entre los grupos.
- H₁: las distribuciones de `CLAGE` presentan diferencias entre los grupos.

Nivel de significación: α = 0,05.

In [7]:
u_stat, p_value = stats.mannwhitneyu(
    clage_0,
    clage_1,
    alternative="two-sided"
)

print(f"Estadístico U: {u_stat:.2f}")
print(f"p-value: {p_value:.6g}")

Estadístico U: 3205297.50
p-value: 1.44581e-44


In [ ]:
### Resultado

La prueba de Mann–Whitney U obtiene un p-value muy inferior a 0,05, por lo
que se rechaza la hipótesis nula.

Existe evidencia estadística de diferencias en la distribución de `CLAGE`
entre los clientes `BAD = 0` y `BAD = 1`.

Este resultado es coherente con el EDA, donde el grupo `BAD = 1` presentó
valores de `CLAGE` inferiores. Sin embargo, la significancia estadística no
indica por sí sola la magnitud o relevancia práctica de la diferencia.

In [8]:
n0 = len(clage_0)
n1 = len(clage_1)

rank_biserial = (2 * u_stat) / (n0 * n1) - 1

print(f"Correlación biserial de rangos: {rank_biserial:.3f}")

Correlación biserial de rangos: 0.271


In [ ]:
### Tamaño del efecto e interpretación

La correlación biserial de rangos obtenida es de 0,271, lo que indica un
tamaño del efecto pequeño-moderado.

Por tanto, el análisis de `CLAGE` muestra que existen diferencias
estadísticamente significativas entre los grupos `BAD = 0` y `BAD = 1`,
y que dichas diferencias presentan una magnitud apreciable, aunque no
especialmente elevada.

En conjunto con el análisis exploratorio, los clientes `BAD = 1` tienden
a presentar valores inferiores de `CLAGE`. Esto sugiere una asociación
entre una menor antigüedad del historial crediticio y la variable objetivo,
sin implicar una relación causal.

In [9]:
stats.mannwhitneyu(clage_0, clage_1)

MannwhitneyuResult(statistic=np.float64(3205297.5), pvalue=np.float64(1.4458052644704147e-44))

In [ ]:
## 4. Comparación estadística de variables numéricas

Una vez desarrollado de forma detallada el contraste para `CLAGE`, se aplica
el mismo enfoque a un conjunto de variables numéricas relevantes.

Se utiliza la prueba de Mann–Whitney U debido a la presencia de asimetrías,
valores extremos y variables de conteo en los datos.

Además del p-value, se calcula la correlación biserial de rangos como medida
del tamaño del efecto.

In [14]:
variables_test = [
    "LOAN",
    "MORTDUE",
    "VALUE",
    "YOJ",
    "DEROG",
    "DELINQ",
    "CLAGE",
    "NINQ",
    "CLNO",
    "DEBTINC"
]

In [11]:
def mann_whitney_analysis(df, variables):
    
    resultados = []

    for variable in variables:
        
        grupo_0 = df.loc[df["BAD"] == 0, variable].dropna()
        grupo_1 = df.loc[df["BAD"] == 1, variable].dropna()

        u_stat, p_value = stats.mannwhitneyu(
            grupo_0,
            grupo_1,
            alternative="two-sided"
        )

        n0 = len(grupo_0)
        n1 = len(grupo_1)

        rank_biserial = (2 * u_stat) / (n0 * n1) - 1

        resultados.append({
            "variable": variable,
            "mediana_BAD_0": grupo_0.median(),
            "mediana_BAD_1": grupo_1.median(),
            "U": u_stat,
            "p_value": p_value,
            "efecto": rank_biserial
        })

    return pd.DataFrame(resultados)

In [15]:
resultados_mw = mann_whitney_analysis(df, variables_test)

resultados_mw.round(4)

,variable,mediana_BAD_0,mediana_BAD_1,U,p_value,efecto
0,LOAN,16900.0000,14900.0000,3280550.0,0.0000,0.1566
1,MORTDUE,66839.0000,60279.0000,2621816.0,0.0000,0.1108
2,VALUE,90659.0000,82000.0000,2876419.5,0.0000,0.1140
3,YOJ,7.0000,6.0000,2624710.5,0.0000,0.0808
4,DEROG,0.0000,0.0000,1755282.5,0.0000,-0.2324
5,DELINQ,0.0000,0.0000,1561803.5,0.0000,-0.3440
6,CLAGE,180.4158,132.8667,3205297.5,0.0000,0.2707
7,NINQ,1.0000,1.0000,1922957.0,0.0000,-0.2038
8,CLNO,20.0000,20.0000,2670473.5,0.2579,0.0216
9,DEBTINC,34.5417,38.0798,603565.0,0.0000,-0.3018


In [ ]:
### Interpretación de los resultados

La prueba de Mann–Whitney muestra evidencia estadística de diferencias entre
`BAD = 0` y `BAD = 1` para la mayoría de las variables analizadas.

Los mayores tamaños del efecto se observan en `DELINQ`, `DEBTINC`, `CLAGE`,
`DEROG` y `NINQ`, lo que coincide con varios de los patrones identificados
durante el análisis exploratorio.

En particular, `DELINQ` presenta la misma mediana (0) en ambos grupos, pero
un tamaño del efecto relevante, mostrando que la comparación de medianas por
sí sola no permite describir completamente las diferencias entre las
distribuciones.

`CLNO` presenta un p-value de 0,2579 y un tamaño del efecto muy reducido
(0,0216), por lo que no se encuentra evidencia estadística suficiente de
diferencias entre los grupos para esta variable.

Estos resultados muestran también la importancia de considerar conjuntamente
la significancia estadística y el tamaño del efecto.

In [ ]:
## 5. Asociación entre variables categóricas y BAD

Para analizar la posible asociación entre las variables categóricas y `BAD`
se utilizará la prueba chi-cuadrado de independencia.

Hipótesis:

- H₀: la variable categórica y `BAD` son independientes.
- H₁: existe asociación entre ambas variables.

Se utilizará α = 0,05.

In [16]:
tabla_job = pd.crosstab(df["JOB"], df["BAD"])

tabla_job

BAD,0,1
JOB,,
Mgr,588,179
Office,823,125
Other,1834,554
ProfExe,1064,212
Sales,71,38
Self,135,58


In [17]:
chi2, p_value, dof, expected = stats.chi2_contingency(tabla_job)

print(f"Chi-cuadrado: {chi2:.4f}")
print(f"p-value: {p_value:.6g}")
print(f"Grados de libertad: {dof}")

Chi-cuadrado: 81.9325
p-value: 3.30668e-16
Grados de libertad: 5


In [18]:
expected

array([[ 609.57665904,  157.42334096],
       [ 753.42721352,  194.57278648],
       [1897.8736138 ,  490.1263862 ],
       [1014.10667136,  261.89332864],
       [  86.62823447,   22.37176553],
       [ 153.38760782,   39.61239218]])

In [ ]:
### Resultado para JOB

La prueba chi-cuadrado obtiene un p-value muy inferior a 0,05, por lo que
se rechaza la hipótesis nula de independencia.

Existe evidencia estadística de asociación entre `JOB` y `BAD`.

Este resultado no implica causalidad y tampoco informa por sí solo sobre
la magnitud de la asociación, por lo que se calculará posteriormente una
medida de tamaño del efecto.

In [19]:
categorical_vars = ["JOB", "REASON"]

resultados_chi2 = []

for variable in categorical_vars:
    
    tabla = pd.crosstab(df[variable], df["BAD"])
    
    chi2, p_value, dof, expected = stats.chi2_contingency(tabla)
    
    n = tabla.to_numpy().sum()
    k = min(tabla.shape)
    
    cramers_v = np.sqrt(chi2 / (n * (k - 1)))
    
    resultados_chi2.append({
        "variable": variable,
        "chi2": chi2,
        "p_value": p_value,
        "grados_libertad": dof,
        "cramers_v": cramers_v
    })

resultados_chi2 = pd.DataFrame(resultados_chi2)

resultados_chi2.round(4)

,variable,chi2,p_value,grados_libertad,cramers_v
0,JOB,81.9325,0.0000,5,0.1201
1,REASON,8.0398,0.0046,1,0.0375


In [ ]:
### Interpretación de las variables categóricas

Las pruebas chi-cuadrado muestran evidencia estadística de asociación entre
`BAD` y las variables `JOB` y `REASON`, ya que ambas presentan p-values
inferiores a 0,05.

Sin embargo, el análisis del tamaño del efecto mediante V de Cramér muestra
que la magnitud de estas asociaciones es limitada.

`JOB` presenta una asociación de magnitud reducida (V = 0,1201), mientras que
`REASON` presenta una asociación muy débil (V = 0,0375).

Estos resultados muestran que la significancia estadística no implica
necesariamente una asociación fuerte o una elevada relevancia predictiva.

In [ ]:
## 6. Conclusiones del análisis estadístico

El análisis inferencial confirma que varias de las diferencias identificadas
durante el EDA presentan evidencia estadística.

Entre las variables numéricas, `DELINQ`, `DEBTINC`, `CLAGE`, `DEROG` y
`NINQ` presentan algunos de los mayores tamaños del efecto. Por el contrario,
`CLNO` no presenta evidencia estadística suficiente de diferencias entre
los grupos `BAD = 0` y `BAD = 1`.

Las variables categóricas `JOB` y `REASON` presentan asociación
estadísticamente significativa con `BAD`, aunque sus tamaños del efecto son
reducidos, especialmente en el caso de `REASON`.

En conjunto, los resultados muestran la importancia de diferenciar entre
significancia estadística y magnitud del efecto. Una relación estadísticamente
significativa no implica necesariamente una elevada relevancia práctica o
predictiva.

El siguiente paso será preparar los datos para construir modelos predictivos
de riesgo de crédito.